# 托卡马克平衡中的磁坐标系

本教程比较磁约束聚变中使用的四种通量坐标系：
**PEST**、**Boozer**、**Hamada** 和 **等弧长坐标**。

## 什么是通量坐标？

在托卡马克中，磁力线位于嵌套的环形曲面上，这些曲面称为*通量面*。
通量坐标系 $(\psi, \theta, \varphi)$ 满足：
- $\psi$ 标记通量面（例如 $\psi_{\rm norm} = 0$ 位于磁轴，$1$ 位于 LCFS）
- $\varphi$ 是标准环向角
- $\theta$ 是按特定有用性质定义的极向角

这四种系统只在 $\theta$ 的选择上不同：

| 系统 | 极向角 $\theta$ 的定义 | 关键性质 |
|--------|-------------------------------------|---------------|
| **PEST** | $\partial(\mathbf{B}\cdot\nabla\theta^*) / \partial\theta^* = 0$：直磁力线 | 最简单的直磁力线坐标 |
| **Boozer** | PEST + Jacobian（雅可比行列式）是通量函数：$\sqrt{g} = \sqrt{g}(\psi)$ | $\mathbf{J}\cdot\nabla\varphi = {\rm const}(\psi)$；常用于仿星器代码与 VMEC 输出 |
| **Hamada** | 从磁轴出发的等面积角 | $\mathbf{J}\cdot\nabla\theta = {\rm const}(\psi)$；简化 MHD 稳定性矩阵 |
| **等弧长** | 沿通量面的弧长在 $\theta_{\rm ea}$ 中均匀 | 数值上稳健；构造简单 |

## 为什么这些选择重要？

- **直磁力线**（PEST、Boozer、Hamada）：环向模数为 $n$、极向模数为 $m$ 的模在满足 $q = m/n$ 的位置发生共振。这会简化 MHD 模的 Fourier 分析。
- **Boozer**：$1/B^2$ 漂移是纯径向的，因此可以简化新经典输运理论和动理学方程。
- **Hamada**：电流流线也为直线，这对某些 MHD 稳定性代码是必要条件。
- **等弧长**：适合数值网格；能够很好地解析等离子体边界。


## 设置：Solov'ev 解析平衡

我们使用按 EAST 尺度缩放的 Cerfon & Freidberg (2010) Solov'ev 平衡（R0 ≈ 1.86 m）。


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec

from pyna.toroidal.equilibrium.Solovev import solovev_iter_like
from pyna.toroidal.coords.PEST import build_PEST_mesh
from pyna.toroidal.coords.EqualArc import build_equal_arc_mesh
from pyna.toroidal.coords.Hamada import build_Hamada_mesh
from pyna.toroidal.coords.Boozer import build_Boozer_mesh

# Create equilibrium (scale=0.3 → EAST-sized, R0≈1.86 m)
eq = solovev_iter_like(scale=0.3)
print(f"Equilibrium: R0={eq.R0:.2f} m, a={eq.a:.2f} m, B0={eq.B0:.1f} T")
print(f"κ={eq.kappa:.2f}, δ={eq.delta:.2f}, q0={eq.q0:.2f}")
Rmaxis, Zmaxis = eq.magnetic_axis
print(f"Magnetic axis: R={Rmaxis:.3f} m, Z={Zmaxis:.3f} m")

Equilibrium: R0=1.86 m, a=0.60 m, B0=5.3 T
κ=1.70, δ=0.33, q0=1.50
Magnetic axis: R=1.946 m, Z=0.000 m


In [2]:
# Build the background grid for the equilibrium
nR, nZ = 150, 150
R_grid = np.linspace(0.3 * eq.R0, 1.5 * eq.R0, nR)
Z_grid = np.linspace(-eq.a * eq.kappa * 1.3, eq.a * eq.kappa * 1.3, nZ)
Rg, Zg = np.meshgrid(R_grid, Z_grid, indexing='ij')

BR_grid, BZ_grid = eq.BR_BZ(Rg, Zg)
BPhi_grid = eq.Bphi(Rg)
psi_norm_grid = eq.psi(Rg, Zg)

# Build PEST mesh
# ns=40 radial surfaces, ntheta=181 poloidal points
ns = 40
ntheta = 181
S, TET, R_mesh, Z_mesh, q_iS = build_PEST_mesh(
    R_grid, Z_grid, BR_grid, BZ_grid, BPhi_grid, psi_norm_grid,
    Rmaxis, Zmaxis, ns=ns, ntheta=ntheta
)
print(f"PEST mesh built: {ns} surfaces, {ntheta} poloidal points")
print(f"S range: [{S[1]:.3f}, {S[-1]:.3f}]")
print(f"q range: [{q_iS[1]:.2f}, {q_iS[-1]:.2f}]")

PEST mesh built: 40 surfaces, 181 poloidal points
S range: [0.028, 0.972]
q range: [1.49, 2.39]


## 第 3 节：等弧长坐标

等弧长角 $\theta_{\rm ea}$ 对每个通量面进行参数化，使沿该面的弧长与 $\theta_{\rm ea}$ 成正比。这是最简单的非平凡坐标变换。


In [3]:
# Build equal-arc mesh
_, TET_ea, R_ea, Z_ea = build_equal_arc_mesh(S, TET, R_mesh, Z_mesh, n_theta=ntheta)
print(f"Equal-arc mesh built: shape {R_ea.shape}")

# Verify arc length uniformity on a mid-radius surface
i_mid = ns // 2
R_s = R_ea[i_mid, :]
Z_s = Z_ea[i_mid, :]
R_loop = R_s[:-1]; Z_loop = Z_s[:-1]  # drop endpoint duplicate
R_closed = np.append(R_loop, R_loop[0])
Z_closed = np.append(Z_loop, Z_loop[0])
ds = np.sqrt(np.diff(R_closed)**2 + np.diff(Z_closed)**2)
print(f"Arc-length segments at surface {i_mid}: mean={ds.mean()*100:.2f} cm, std={ds.std()*100:.3f} cm")
print(f"  Uniformity (std/mean): {ds.std()/ds.mean():.4f}")

Equal-arc mesh built: shape (40, 181)
Arc-length segments at surface 20: mean=1.27 cm, std=0.000 cm
  Uniformity (std/mean): 0.0001


In [4]:
# Plot equal-arc coordinates
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: R-Z view of flux surfaces with equal-arc mesh lines
ax = axes[0]
# Plot psi_norm contours as background
cs = ax.contour(Rg, Zg, psi_norm_grid, levels=np.linspace(0, 1, 11), 
                colors='lightgray', linewidths=0.5)
# Plot every 5th surface in equal-arc
stride = max(1, ns // 10)
colors = cm.plasma(np.linspace(0.2, 0.9, ns // stride + 1))
for k, i in enumerate(range(1, ns, stride)):
    ax.plot(R_ea[i, :], Z_ea[i, :], color=colors[k], lw=1.0)
# Plot a few poloidal lines (constant θ_ea)
for j in range(0, ntheta-1, ntheta // 8):
    ax.plot(R_ea[1:, j], Z_ea[1:, j], 'b-', lw=0.5, alpha=0.5)
ax.plot(Rmaxis, Zmaxis, 'r+', ms=10, mew=2, label='Axis')
ax.set_xlabel('R (m)')
ax.set_ylabel('Z (m)')
ax.set_title('Equal-arc mesh (R-Z view)')
ax.set_aspect('equal')
ax.legend()

# Right: arc-length increments as function of theta_ea
ax = axes[1]
for k, i in enumerate(range(2, ns-1, stride)):
    R_s = R_ea[i, :-1]; Z_s = Z_ea[i, :-1]
    R_c = np.append(R_s, R_s[0]); Z_c = np.append(Z_s, Z_s[0])
    ds_s = np.sqrt(np.diff(R_c)**2 + np.diff(Z_c)**2)
    ds_s_norm = ds_s / ds_s.mean()
    ax.plot(TET_ea[:-1], ds_s_norm, color=colors[k], lw=0.8, 
            label=f'S={S[i]:.2f}' if k % 2 == 0 else None)
ax.axhline(1.0, color='k', ls='--', lw=1, label='Ideal (uniform)')
ax.set_xlabel(r'$\theta_{\rm ea}$ (rad)')
ax.set_ylabel('Arc-length increment / mean')
ax.set_title('Arc-length uniformity in equal-arc coordinates')
ax.set_xlim(0, 2*np.pi)
ax.legend(fontsize=8, ncol=2)

plt.tight_layout()
plt.savefig('equal_arc_coords.png', dpi=100, bbox_inches='tight')
plt.show()
print("Equal-arc: arc-length segments are uniform to within ~1% (interpolation discretisation)")

Equal-arc: arc-length segments are uniform to within ~1% (interpolation discretisation)


## 第 4 节：PEST 直磁力线坐标

在 PEST 坐标中，安全因子 $q(\psi)$ 满足：
$$q(\psi) = \frac{\mathbf{B}\cdot\nabla\varphi}{\mathbf{B}\cdot\nabla\theta^*} = {\rm const \; on \; each \; surface}$$

这意味着磁力线在 $(\theta^*, \varphi)$ 平面中表现为斜率为 $1/q$ 的**直线**。


In [5]:
# Show q(S) profile from PEST integration
q_valid = q_iS[1:]
S_valid = S[1:]

# Compare with analytic equilibrium q
psi_vals = S_valid**2
q_analytic = eq.q_profile(psi_vals, n_theta=256)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot(S_valid, q_valid, 'b-o', ms=3, label='PEST q(S)')
finite = np.isfinite(q_analytic)
ax.plot(S_valid[finite], q_analytic[finite], 'r--', label='Analytic q(S)')
ax.set_xlabel('S = √ψ_norm')
ax.set_ylabel('Safety factor q')
ax.set_title('Safety factor profile: PEST vs analytic')
ax.legend()
ax.grid(True, alpha=0.3)

# PEST mesh in R-Z
ax = axes[1]
cs = ax.contour(Rg, Zg, psi_norm_grid, levels=np.linspace(0, 1, 11),
                colors='lightgray', linewidths=0.5)
stride_s = max(1, ns // 10)
colors_pest = cm.viridis(np.linspace(0.2, 0.9, ns // stride_s + 1))
for k, i in enumerate(range(1, ns, stride_s)):
    ax.plot(R_mesh[i, :], Z_mesh[i, :], color=colors_pest[k], lw=1.0)
for j in range(0, ntheta-1, ntheta // 8):
    ax.plot(R_mesh[1:, j], Z_mesh[1:, j], 'g-', lw=0.5, alpha=0.5)
ax.plot(Rmaxis, Zmaxis, 'r+', ms=10, mew=2)
ax.set_xlabel('R (m)')
ax.set_ylabel('Z (m)')
ax.set_title('PEST mesh (R-Z view)')
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('pest_coords.png', dpi=100, bbox_inches='tight')
plt.show()

In [6]:
# Demonstrate straight field lines in PEST theta-phi space
# In PEST: a field line traces theta* = phi/q + const
fig, ax = plt.subplots(figsize=(8, 5))

phi_line = np.linspace(0, 4 * np.pi, 200)
surface_indices = [ns // 5, ns // 3, ns // 2, 2 * ns // 3, 4 * ns // 5]
colors_fl = cm.Set1(np.linspace(0, 0.8, len(surface_indices)))

for k, i_s in enumerate(surface_indices):
    if i_s >= ns or not np.isfinite(q_iS[i_s]):
        continue
    q_s = q_iS[i_s]
    # Field line: theta* = phi / q (PEST straight field line)
    theta_fl = (phi_line / q_s) % (2 * np.pi)
    ax.plot(phi_line / np.pi, theta_fl / np.pi, color=colors_fl[k], 
            label=f'S={S[i_s]:.2f}, q={q_s:.2f}', lw=1.5)

ax.set_xlabel(r'$\varphi / \pi$')
ax.set_ylabel(r'$\theta^* / \pi$ (PEST)')
ax.set_title('Field lines in PEST $(\\theta^*, \\varphi)$ space — straight lines with slope $1/q$')
ax.legend(fontsize=9)
ax.set_xlim(0, 4)
ax.set_ylim(0, 2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pest_fieldlines.png', dpi=100, bbox_inches='tight')
plt.show()

## 第 5 节：Boozer 坐标

Boozer 坐标在 PEST 的基础上进一步要求 Jacobian（雅可比行列式）$\sqrt{g}$ 是通量面量（只依赖 $\psi$，不依赖 $\theta$）。这保证 $\mathbf{J}\cdot\nabla\varphi = {\rm const}(\psi)$。

**构造**：$\theta_B = \theta^* + \lambda(\psi, \theta^*)$，其中
$$\frac{\partial\lambda}{\partial\theta^*} = \frac{\langle\sqrt{g}\rangle}{\sqrt{g}} - 1$$
这里 $\langle\cdot\rangle$ 表示通量面平均 $\frac{1}{2\pi}\int_0^{2\pi}\cdot\,d\theta^*$。


In [7]:
# Build Boozer mesh
_, TET_B, R_B, Z_B, lambda_corr = build_Boozer_mesh(
    S, TET, R_mesh, Z_mesh, q_iS, equilibrium=eq, n_theta=ntheta
)
print(f"Boozer mesh built: shape {R_B.shape}")
print(f"Max |λ| correction: {np.nanmax(np.abs(lambda_corr)):.4f} rad")

# Show the angle correction lambda(psi, theta*)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
# lambda correction as a function of theta for several surfaces
for k, i_s in enumerate(range(2, ns-1, max(1, ns // 8))):
    lam = lambda_corr[i_s, :]
    if np.any(np.isfinite(lam)):
        ax.plot(TET / np.pi, lam, label=f'S={S[i_s]:.2f}', 
                color=cm.plasma(0.1 + 0.8 * i_s / ns))
ax.set_xlabel(r'$\theta^* / \pi$ (PEST)')
ax.set_ylabel(r'$\lambda = \theta_B - \theta^*$ (rad)')
ax.set_title('Boozer angle correction $\\lambda(\\psi, \\theta^*)$')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[1]
cs = ax.contour(Rg, Zg, psi_norm_grid, levels=np.linspace(0, 1, 11),
                colors='lightgray', linewidths=0.5)
stride_s = max(1, ns // 10)
colors_b = cm.inferno(np.linspace(0.2, 0.9, ns // stride_s + 1))
for k, i in enumerate(range(1, ns, stride_s)):
    ax.plot(R_B[i, :], Z_B[i, :], color=colors_b[k], lw=1.0)
for j in range(0, ntheta-1, ntheta // 8):
    ax.plot(R_B[1:, j], Z_B[1:, j], 'm-', lw=0.5, alpha=0.5)
ax.plot(Rmaxis, Zmaxis, 'r+', ms=10, mew=2)
ax.set_xlabel('R (m)')
ax.set_ylabel('Z (m)')
ax.set_title('Boozer mesh (R-Z view)')
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('boozer_coords.png', dpi=100, bbox_inches='tight')
plt.show()

Boozer mesh built: shape (40, 181)
Max |λ| correction: 0.5429 rad


## 第 6 节：Hamada 坐标

Hamada 坐标要求磁力线和电流密度线都为直线：
$$\mathbf{J}\cdot\nabla\psi = 0, \quad \mathbf{J}\cdot\nabla\theta_H = 0, \quad \mathbf{J}\cdot\nabla\varphi_H = {\rm const}(\psi)$$

对于轴对称平衡，这等价于把极向角做成**等面积角**：从磁轴扫到角度 $\theta_H$ 的面积与 $\theta_H$ 成正比。


In [8]:
# Build Hamada mesh
_, TET_H, R_H, Z_H = build_Hamada_mesh(S, TET, R_mesh, Z_mesh, n_theta=ntheta)
print(f"Hamada mesh built: shape {R_H.shape}")

# Verify equal-area property
R_ax = R_mesh[0, 0]
Z_ax = Z_mesh[0, 0]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
# Show cumulative area vs theta_H for several surfaces
for k, i_s in enumerate(range(2, ns-2, max(1, ns // 8))):
    R_s = R_H[i_s, :-1]; Z_s = Z_H[i_s, :-1]
    R_c = np.append(R_s, R_s[0]); Z_c = np.append(Z_s, Z_s[0])
    dA = 0.5 * ((R_c[:-1] - R_ax) * (Z_c[1:] - Z_ax) - 
                 (R_c[1:] - R_ax) * (Z_c[:-1] - Z_ax))
    A_cum = np.cumsum(dA)
    A_total = A_cum[-1]
    if abs(A_total) > 1e-10:
        ax.plot(TET_H[:-1] / np.pi, A_cum / A_total, 
                color=cm.cool(0.1 + 0.8 * i_s / ns),
                label=f'S={S[i_s]:.2f}')
# Ideal line
ax.plot([0, 2], [0, 1], 'k--', lw=2, label='Ideal (linear)')
ax.set_xlabel(r'$\theta_H / \pi$ (Hamada)')
ax.set_ylabel('Cumulative area / total')
ax.set_title('Hamada: cumulative area is linear in $\\theta_H$')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

ax = axes[1]
cs = ax.contour(Rg, Zg, psi_norm_grid, levels=np.linspace(0, 1, 11),
                colors='lightgray', linewidths=0.5)
colors_h = cm.cool(np.linspace(0.2, 0.9, ns // stride_s + 1))
for k, i in enumerate(range(1, ns, stride_s)):
    ax.plot(R_H[i, :], Z_H[i, :], color=colors_h[k], lw=1.0)
for j in range(0, ntheta-1, ntheta // 8):
    ax.plot(R_H[1:, j], Z_H[1:, j], 'c-', lw=0.5, alpha=0.5)
ax.plot(Rmaxis, Zmaxis, 'r+', ms=10, mew=2)
ax.set_xlabel('R (m)')
ax.set_ylabel('Z (m)')
ax.set_title('Hamada mesh (R-Z view)')
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig('hamada_coords.png', dpi=100, bbox_inches='tight')
plt.show()

Hamada mesh built: shape (40, 181)


## 第 7 节：比较面板：四种坐标系

这里在 R-Z 平面中叠加同一组通量面的四种网格。


In [9]:
fig = plt.figure(figsize=(16, 14))
gs = GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)

meshes = [
    ('PEST', TET, R_mesh, Z_mesh, 'viridis', 'g'),
    ('Equal-arc', TET_ea, R_ea, Z_ea, 'plasma', 'b'),
    ('Boozer', TET_B, R_B, Z_B, 'inferno', 'm'),
    ('Hamada', TET_H, R_H, Z_H, 'cool', 'c'),
]

stride_s = max(1, ns // 8)
stride_t = ntheta // 12

for idx, (name, tet, R_m, Z_m, cmap, pline_color) in enumerate(meshes):
    ax = fig.add_subplot(gs[idx // 2, idx % 2])
    
    # Flux surface contours as background
    ax.contour(Rg, Zg, psi_norm_grid, levels=np.linspace(0.05, 0.95, 10),
               colors='lightgray', linewidths=0.4)
    
    # Mesh surfaces (constant S)
    surface_colors = plt.get_cmap(cmap)(np.linspace(0.2, 0.9, ns // stride_s + 1))
    for k, i in enumerate(range(1, ns, stride_s)):
        ax.plot(R_m[i, :], Z_m[i, :], color=surface_colors[k], lw=1.2)
    
    # Poloidal lines (constant theta)
    for j in range(0, len(tet)-1, stride_t):
        ax.plot(R_m[1:, j], Z_m[1:, j], color=pline_color, 
                lw=0.6, alpha=0.6)
    
    ax.plot(Rmaxis, Zmaxis, 'r+', ms=12, mew=2)
    ax.set_xlabel('R (m)', fontsize=11)
    ax.set_ylabel('Z (m)', fontsize=11)
    ax.set_title(f'{name} coordinates', fontsize=13, fontweight='bold')
    ax.set_aspect('equal')
    
    # Add annotation
    ann = {'PEST': 'Straight B field lines',
           'Equal-arc': 'Uniform arc length',
           'Boozer': 'Straight B + uniform Jacobian',
           'Hamada': 'Straight B + equal area'}[name]
    ax.text(0.05, 0.95, ann, transform=ax.transAxes, fontsize=9,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

fig.suptitle('Comparison of Magnetic Coordinate Systems\n(coloured lines = flux surfaces, thin lines = poloidal mesh)', 
             fontsize=14, y=1.01)
plt.savefig('coords_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## 第 8 节：物理解释

### 8.1 Fourier 模耦合

在直磁力线坐标（PEST、Boozer、Hamada）中，单一环向模数 $n$ 的扰动会在 $q = m/n$ 的通量面发生共振。关于 $\theta$ 的 Fourier 分解具有清晰的模结构。

相比之下，如果使用几何角（或等弧长角），$(m,n)$ 中的单一模会表现为**多个** Fourier 分量，并耦合不同的 $m$ 值，这就是所谓的 Fourier 耦合问题。


In [10]:
# Demonstrate: Fourier spectrum of R(theta) — 2D heatmap (m vs S) for PEST and equal-arc

m_show = 12   # show modes m=0..m_show-1

# Compute FFT of R(theta) - <R> at each surface for both coordinate systems
S_arr   = np.linspace(0.1, 0.9, ns)  # approx normalised flux label per surface
fft_PEST = np.zeros((ns, m_show))
fft_EA   = np.zeros((ns, m_show))

for i in range(ns):
    R_pest = R_mesh[i, :-1] - R_mesh[i, :-1].mean()
    R_ea_i = R_ea[i, :-1]   - R_ea[i, :-1].mean()
    n_pts_pest = len(R_pest)
    n_pts_ea   = len(R_ea_i)

    fft_p = np.abs(np.fft.rfft(R_pest))[:m_show]
    fft_p = fft_p / (fft_p.max() + 1e-30)
    fft_PEST[i, :len(fft_p)] = fft_p[:m_show]

    fft_e = np.abs(np.fft.rfft(R_ea_i))[:m_show]
    fft_e = fft_e / (fft_e.max() + 1e-30)
    fft_EA[i, :len(fft_e)] = fft_e[:m_show]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, data, name in zip(
    axes,
    [fft_PEST, fft_EA],
    ['PEST (straight field line)', 'Equal-arc θ'],
):
    # log scale heatmap: rows = radial surface index, cols = mode m
    log_data = np.log10(data + 1e-6)
    im = ax.imshow(
        log_data.T,          # shape (m_show, ns): mode vs surface
        origin='lower',
        aspect='auto',
        cmap='hot_r',
        vmin=log_data.max() - 3,   # show 3 decades
        vmax=log_data.max(),
        extent=[S_arr[0], S_arr[-1], -0.5, m_show - 0.5],
        interpolation='nearest',
    )
    cbar = fig.colorbar(im, ax=ax, pad=0.02, shrink=0.85)
    cbar.set_label(r'$\log_{10}$ normalised FFT amplitude', fontsize=8)
    ax.set_xlabel(r'Normalised flux label $S$', fontsize=10)
    ax.set_ylabel('Poloidal mode m', fontsize=10)
    ax.set_yticks(np.arange(0, m_show, 2))
    ax.set_title(f'R(θ) Fourier spectrum: {name}', fontsize=11)

# Annotation: PEST should be concentrated at m=1; equal-arc spreads
axes[0].text(0.05, 0.9, 'Spectrum concentrated\nat m=1 (ideal)', 
             transform=axes[0].transAxes, fontsize=8, color='white',
             bbox=dict(boxstyle='round', facecolor='steelblue', alpha=0.5))
axes[1].text(0.05, 0.9, 'Spectrum spreads to\nhigher m (non-ideal)', 
             transform=axes[1].transAxes, fontsize=8, color='white',
             bbox=dict(boxstyle='round', facecolor='tomato', alpha=0.5))

plt.suptitle('Fourier Content: PEST vs Equal-arc Coordinates', fontsize=12)
plt.tight_layout()
plt.show()

### 8.2 汇总表


In [11]:
summary = """
╔══════════════╦═══════════════════════╦══════════════════════════════╦══════════════════════════╗
║ Coordinate   ║ Definition of θ       ║ Main advantage               ║ Used in                  ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ PEST         ║ Straight B field lines║ Minimal Fourier coupling      ║ PEST, GATO, ELITE codes  ║
║              ║ B·∇θ*/B·∇φ = q(ψ)   ║ q = m/n at resonance         ║                          ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ Boozer       ║ PEST + √g = √g(ψ)    ║ 1/B² drift is purely radial  ║ private stellarator, private stellarator, VMEC output   ║
║              ║ (uniform Jacobian)    ║ Cleaner neoclassical theory   ║                          ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ Hamada       ║ Equal area from axis  ║ J·∇θ = const, MHD stability  ║ TERPSICHORE, CASTOR      ║
║              ║ ∝ swept poloidal area ║ matrices simplified           ║                          ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ Equal-arc    ║ Uniform arc length    ║ Simple construction           ║ Numerical grids, FEM     ║
║              ║ ds/dθ_ea = const     ║ Resolves boundary well        ║                          ║
╚══════════════╩═══════════════════════╩══════════════════════════════╩══════════════════════════╝
"""
print(summary)


╔══════════════╦═══════════════════════╦══════════════════════════════╦══════════════════════════╗
║ Coordinate   ║ Definition of θ       ║ Main advantage               ║ Used in                  ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ PEST         ║ Straight B field lines║ Minimal Fourier coupling      ║ PEST, GATO, ELITE codes  ║
║              ║ B·∇θ*/B·∇φ = q(ψ)   ║ q = m/n at resonance         ║                          ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ Boozer       ║ PEST + √g = √g(ψ)    ║ 1/B² drift is purely radial  ║ private stellarator, private stellarator, VMEC output   ║
║              ║ (uniform Jacobian)    ║ Cleaner neoclassical theory   ║                          ║
╠══════════════╬═══════════════════════╬══════════════════════════════╬══════════════════════════╣
║ Hamada       ║ Equal area from axis  ║ J·∇θ = const, MHD stability  ║ TERPSI

### 8.3 各系统之间的关系

四种系统都可通过形如 $\theta_{\rm new} = \theta^* + f(\psi, \theta^*)$ 的角度变换彼此联系：

- **等弧长** → 等弧长参数化（纯几何）
- **PEST** → 等磁力线绕转（需要积分 $B_R, B_Z$）
- **Boozer** → PEST + 用周期修正压平 Jacobian（雅可比行列式）
- **Hamada** → 等面积变换（涉及包围面积，等价于使 $\sqrt{g}$ 与总表面积成正比）

对于托卡马克，Boozer 与 Hamada 通常非常相似，因为这两者涉及的通量面平均量（Jacobian 与面积）都受同一压力平衡控制。


In [12]:
# Final comparison: all four theta angles for a single field line
# Show how theta_PEST, theta_B, theta_H, theta_ea relate on the midradius surface
i_surf = ns // 2
print(f"Comparing poloidal angle representations on surface S={S[i_surf]:.3f}")
print(f"  Safety factor q = {q_iS[i_surf]:.3f}")

# For each coordinate system, compute the geometric angle (atan2(Z-Zax, R-Rax))
fig, ax = plt.subplots(figsize=(10, 5))

for name, tet_arr, R_m, Z_m, color in [
    ('PEST', TET, R_mesh, Z_mesh, 'blue'),
    ('Equal-arc', TET_ea, R_ea, Z_ea, 'green'),
    ('Boozer', TET_B, R_B, Z_B, 'red'),
    ('Hamada', TET_H, R_H, Z_H, 'purple'),
]:
    R_s = R_m[i_surf, :]
    Z_s = Z_m[i_surf, :]
    theta_geom = np.arctan2(Z_s - Zmaxis, R_s - Rmaxis) % (2 * np.pi)
    ax.plot(tet_arr / np.pi, theta_geom / np.pi, label=name, color=color, lw=1.5)

# Diagonal = geometric angle equals coordinate angle
ax.plot([0, 2], [0, 2], 'k--', lw=1, alpha=0.5, label='Identity (geom = coord)')
ax.set_xlabel(r'Coordinate angle $\theta / \pi$')
ax.set_ylabel(r'Geometric angle $\theta_{\rm geom} / \pi$')
ax.set_title('Geometric angle vs coordinate angle for each system\n' +
             f'(surface S={S[i_surf]:.2f}, q={q_iS[i_surf]:.2f})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2)
ax.set_ylim(0, 2)

plt.tight_layout()
plt.savefig('theta_comparison.png', dpi=100, bbox_inches='tight')
plt.show()
print("All four systems differ only in how θ is distributed around the flux surface.")

Comparing poloidal angle representations on surface S=0.474
  Safety factor q = 1.638
All four systems differ only in how θ is distributed around the flux surface.
